# 🧪 EXERCISE: Session Gaze Metrics

> **Instructions:** This notebook has intentional gaps — paths, loops, calculations and plot labels that you must complete. Look for `# TODO` and `???` markers. Run each cell in order once you have filled in the blanks.

## Learning objectives

- Set file paths and understand the Pupil Labs data structure
- Filter gaze topics from raw .pldata binary data
- Implement instantaneous velocity calculation for saccade detection
- Add meaningful axis labels and titles to metric plots
- Export trial-level metrics to a CSV file

---

# VISION Task — Single Session: Fixation, Saccade & Blink Metrics

Processes raw Pupil Labs recording data (`.pldata` files) for **one participant, one recording session** of the **VISION arrow-key task** (Aloufi et al. 2021).  
Produces trial-level and session-level eye-movement metrics:

| Metric class | Measures |
|---|---|
| **Fixations** | Count, mean duration, total fixation time, dispersion |
| **Saccades** | Count, amplitude, peak velocity, latency to first saccade |
| **Blinks** | Count, mean duration, rate (blinks/min) |
| **Gaze at response** | Where gaze was located when the arrow key was pressed |

**Data sources:**
- Pupil Labs Core recording folder — `gaze.pldata`, `fixations.pldata`, `blinks.pldata`, `annotation.pldata`
- PsychoPy CSV (optional) — trial-level behavioural data for correlation

**Annotation format produced by VISION_task.py:**
- Onset: `trial_001_onset_ax0_pos3R_circle`  
- Offset: `trial_001_offset_correct` / `trial_001_offset_incorrect`

> Set `RECORDING_DIR` and `PARTICIPANT_ID` in Cell 2, then **Run All**.


---
## 0 Setup

In [ ]:
# SCOPE: one participant × one recording session

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
import msgpack, struct

sns.set_theme(style='darkgrid', palette='tab10', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})

# ── TODO 1: Set your recording directory ──────────────────────────────────────
# Point this to the folder that contains your Pupil Labs .pldata files.
RECORDING_DIR = Path('???')   # e.g. Path('/Users/yourname/recordings/session_001')

PARTICIPANT_ID    = '???'     # e.g. 'JF'
SESSION_LABEL     = '???'     # e.g. 'S001'
GAZE_TOPIC        = 'gaze.3d.01.'
VELOCITY_THRESHOLD_DEG_S = 30
MIN_SACCADE_DUR_S        = 0.010
MAX_SACCADE_DUR_S        = 0.100
BLINK_GROUP_GAP_S        = 0.075
POSITION_STEP_DEG        = 4.5


---
## 1 Load Pupil Labs Data

In [ ]:
def load_pldata(path):
    """Load a Pupil Labs .pldata file -> list of (topic, dict) tuples."""
    records = []
    with open(path, 'rb') as f:
        unpacker = msgpack.Unpacker(f, raw=False, strict_map_key=False)
        for item in unpacker:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                topic, payload = item
                rec = payload if isinstance(payload, dict) else (
                    msgpack.unpackb(payload, raw=False, strict_map_key=False)
                    if isinstance(payload, bytes) else None
                )
                if rec is not None:
                    records.append((str(topic), rec))
    return records


# Screen geometry -- MUST match VISION_task.py constants
MONITOR_WIDTH_CM    = 70.8   # <- edit to match VISION_task.py
MONITOR_HEIGHT_CM   = 39.8   # <- edit to match VISION_task.py
VIEWING_DISTANCE_CM = 90.0   # <- edit to match VISION_task.py
SCREEN_W_DEG = 2 * np.degrees(np.arctan(MONITOR_WIDTH_CM  / (2.0 * VIEWING_DISTANCE_CM)))
SCREEN_H_DEG = 2 * np.degrees(np.arctan(MONITOR_HEIGHT_CM / (2.0 * VIEWING_DISTANCE_CM)))
print(f'Screen: {SCREEN_W_DEG:.1f} deg wide x {SCREEN_H_DEG:.1f} deg tall at {VIEWING_DISTANCE_CM} cm')

def norm_to_deg(norm_x, norm_y):
    """Convert Pupil Labs norm_pos to screen-centred degrees.
    World-camera records : norm_pos in [0, 1]  -> degrees within screen bounds.
    Per-eye records      : norm_pos in eye-camera space -> may produce large values.
    """
    return (norm_x - 0.5) * SCREEN_W_DEG, (norm_y - 0.5) * SCREEN_H_DEG

def is_valid_norm(nx, ny, margin=0.05):
    """True if norm_pos is within the screen (with a small margin for edge gaze)."""
    return (-margin <= nx <= 1 + margin) and (-margin <= ny <= 1 + margin)

def gaze_source_label(topic):
    """Classify a Pupil Labs gaze topic: 'eye0', 'eye1', or 'combined'."""
    t = str(topic).lower()
    if '.0.' in t or t.endswith('.0') or t == '0':
        return 'eye0'
    if '.1.' in t or t.endswith('.1') or t == '1':
        return 'eye1'
    return 'combined'   # binocular / world-camera


# Load ALL gaze records: world-camera AND per-eye
# ---------------------------------------------------
# Records are NOT filtered.  A 'gaze_source' column labels each:
#   combined  -> binocular/world-camera; norm_pos in [0,1]; x_deg/y_deg in screen bounds
#   eye0/eye1 -> per-eye camera space;  norm_pos may be outside [0,1]; x_deg/y_deg can be large
# Use 'in_screen_space = True' to restrict to screen-position analysis downstream.
gaze_path = RECORDING_DIR / 'gaze.pldata'
gaze_raw  = load_pldata(gaze_path)

gaze_rows = []
for topic, r in gaze_raw:
    norm = r.get('norm_pos', [np.nan, np.nan])
    nx, ny = norm[0], norm[1]
    xd, yd = norm_to_deg(nx, ny)
    src    = gaze_source_label(topic)
    gaze_rows.append({
        'timestamp'      : r.get('timestamp', np.nan),
        'confidence'     : r.get('confidence', 0),
        'topic'          : topic,
        'gaze_source'    : src,
        'norm_x'         : nx,
        'norm_y'         : ny,
        'x_deg'          : xd,
        'y_deg'          : yd,
        'in_screen_space': is_valid_norm(nx, ny),
    })

gaze_df = pd.DataFrame(gaze_rows)
gaze_df.sort_values('timestamp', inplace=True)
gaze_df.reset_index(drop=True, inplace=True)

print(f'All gaze records loaded: {len(gaze_df):,}')
for src, grp in gaze_df.groupby('gaze_source'):
    in_scr = grp['in_screen_space'].sum()
    xmin = grp['x_deg'].min(); xmax = grp['x_deg'].max()
    print(f'  {src:<10}: {len(grp):>6,} records  |  in-screen: {in_scr:>6,}  |  x_deg [{xmin:.1f}, {xmax:.1f} deg]')


# Load ALL fixation records (world-camera AND per-eye)
fix_path = RECORDING_DIR / 'fixations.pldata'
fix_raw  = load_pldata(fix_path)
fix_rows = []
for topic, r in fix_raw:
    norm = r.get('norm_pos', [np.nan, np.nan])
    nx, ny = norm[0], norm[1]
    raw_dur = r.get('duration', 0)
    # Auto-detect unit: Pupil Labs >= 2019 stores duration in seconds (< 10);
    # very old versions used ms (>> 10 for typical fixations).
    dur_ms = raw_dur * 1000 if 10 > raw_dur > 0.1 else raw_dur
    xd, yd = norm_to_deg(nx, ny)
    src    = gaze_source_label(topic)
    fix_rows.append({
        'timestamp'      : r.get('timestamp', np.nan),
        'duration_ms'    : dur_ms,
        'topic'          : topic,
        'gaze_source'    : src,
        'norm_x'         : nx,
        'norm_y'         : ny,
        'x_deg'          : xd,
        'y_deg'          : yd,
        'dispersion'     : r.get('dispersion', np.nan),
        'confidence'     : r.get('confidence', 0),
        'in_screen_space': is_valid_norm(nx, ny),
    })

fix_df = pd.DataFrame(fix_rows)
if len(fix_df):
    fix_df.sort_values('timestamp', inplace=True)
    fix_df.reset_index(drop=True, inplace=True)
    print(f'\nAll fixation records: {len(fix_df):,}')
    for src, grp in fix_df.groupby('gaze_source'):
        in_scr = grp['in_screen_space'].sum()
        print(f'  {src:<10}: {len(grp):>6,} records  |  in-screen: {in_scr:>6,}')
    combined_fix = fix_df[fix_df['in_screen_space']]
    if len(combined_fix):
        print(f'\n  In-screen fixation durations (ms):')
        print(f'    min:{combined_fix["duration_ms"].min():.0f}  '
              f'median:{combined_fix["duration_ms"].median():.0f}  '
              f'max:{combined_fix["duration_ms"].max():.0f}  '
              f'mean:{combined_fix["duration_ms"].mean():.0f}')
        n_short = (combined_fix['duration_ms'] < MIN_FIX_DURATION_MS).sum()
        print(f'    Micro-fixations < {MIN_FIX_DURATION_MS} ms: {n_short}')


# Blinks
blink_path = RECORDING_DIR / 'blinks.pldata'
blink_raw  = load_pldata(blink_path)
blink_df   = pd.DataFrame([{
    'timestamp' : r.get('timestamp', np.nan),
    'type'      : r.get('type', ''),
    'confidence': r.get('confidence', 0),
} for _, r in blink_raw])


# Annotations
ann_path = RECORDING_DIR / 'annotation.pldata'
ann_raw  = load_pldata(ann_path)
ann_df   = pd.DataFrame([{
    'timestamp' : r.get('timestamp', np.nan),
    'label'     : r.get('label', ''),
} for _, r in ann_raw]).sort_values('timestamp').reset_index(drop=True)

print(f'\nBlink events   : {len(blink_df):,}')
print(f'Annotations    : {len(ann_df):,}')
print(f'\nSample annotations:')
print(ann_df['label'].head(12).to_string())

In [ ]:
# ── TODO 2: Load the gaze data from the Pupil Labs recording ────────────────
# pldata_all contains (topic, data_dict) tuples loaded by load_pldata().
# Filter only records where topic starts with GAZE_TOPIC ('gaze.3d.01.')
# and build a DataFrame with at least: 'timestamp', 'norm_pos_x', 'norm_pos_y', 'confidence'

# Your code here:
gaze_records = ???   # hint: list comprehension over pldata_all

gaze_df = pd.DataFrame(???)  # build DataFrame from filtered records
gaze_df = gaze_df.sort_values('timestamp').reset_index(drop=True)
print(f'Gaze samples loaded: {len(gaze_df)}')
gaze_df.head(3)


In [ ]:
fix_raw

In [ ]:
# ── Raw gaze trace — what does the eye-tracker actually record? ─────────────
# gaze_df is already loaded from the .pldata file above — no CSV needed.
from scipy.ndimage import gaussian_filter1d

if 'gaze_df' not in dir() or gaze_df is None or len(gaze_df) == 0:
    print("⚠  gaze_df not found — run the data-loading cells above first.")
else:
    t_min = gaze_df['timestamp'].min()
    t_max = min(t_min + 30, gaze_df['timestamp'].max())  # first 30 seconds

    g_win = gaze_df[(gaze_df['timestamp'] >= t_min) &
                    (gaze_df['timestamp'] <= t_max)].copy()
    g_win['t_rel'] = g_win['timestamp'] - t_min

    # Smooth gaze position using Gaussian kernel (~5 ms)
    sr    = 1.0 / np.median(np.diff(gaze_df['timestamp'].values))
    sigma = max(1, int(0.005 * sr))
    xs_sm = gaussian_filter1d(g_win['x_deg'].values.astype(float), sigma)
    ys_sm = gaussian_filter1d(g_win['y_deg'].values.astype(float), sigma)

    # Compute instantaneous gaze velocity (degrees/second)
    dt      = np.diff(g_win['timestamp'].values)
    vel_win = np.hypot(np.diff(xs_sm), np.diff(ys_sm)) / np.where(dt > 0, dt, 1e-6)
    vel_win = np.append(vel_win, vel_win[-1])

    t = g_win['t_rel'].values

    # ── 3-panel dark-background figure ──────────────────────────────────────
    fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True, facecolor='#1a1a1a')
    for ax in axes:
        ax.set_facecolor('#1a1a1a')
        for spine in ax.spines.values():
            spine.set_color('white'); spine.set_linewidth(0.8)
        ax.tick_params(colors='white', labelsize=9)
        ax.xaxis.label.set_color('white')
        ax.yaxis.label.set_color('white')
        ax.grid(True, alpha=0.2, color='white', linestyle='-', linewidth=0.3)

    # Panel 1 — Gaze position
    axes[0].plot(t, xs_sm, lw=0.9, color='steelblue',  label='X (horizontal)')
    axes[0].plot(t, ys_sm, lw=0.9, color='darkorange', label='Y (vertical)')
    axes[0].set_ylabel('Gaze position (°)', color='white', fontweight='bold', fontsize=10)
    axes[0].legend(loc='upper right', fontsize=9, facecolor='#1a1a1a',
                   edgecolor='white', labelcolor='white')

    # Panel 2 — Eye velocity with threshold
    axes[1].plot(t, vel_win, lw=0.7, color='seagreen')
    axes[1].axhline(VELOCITY_THRESHOLD_DEG_S, color='red', ls='--', lw=0.8,
                    label=f'{VELOCITY_THRESHOLD_DEG_S}°/s threshold')
    axes[1].set_ylabel('Velocity (°/s)', color='white', fontweight='bold', fontsize=10)
    axes[1].set_ylim(0, min(vel_win.max(), 800))
    axes[1].legend(loc='upper right', fontsize=9, facecolor='#1a1a1a',
                   edgecolor='white', labelcolor='white')

    # Panel 3 — Confidence
    axes[2].plot(t, g_win['confidence'].values, lw=0.7, color='mediumpurple')
    axes[2].axhline(MIN_GAZE_CONFIDENCE, color='red', ls='--', lw=0.8,
                    label=f'Min confidence {MIN_GAZE_CONFIDENCE}')
    axes[2].set_ylabel('Gaze confidence', color='white', fontweight='bold', fontsize=10)
    axes[2].set_xlabel('Time from experiment start (s)', color='white', fontweight='bold', fontsize=10)
    axes[2].set_ylim(0, 1.05)
    axes[2].legend(loc='upper right', fontsize=9, facecolor='#1a1a1a',
                   edgecolor='white', labelcolor='white')

    fig.suptitle(f'{PARTICIPANT_ID} — Gaze trace (first 30 s)',
                 color='white', fontweight='bold', fontsize=13, y=0.98)
    plt.tight_layout()
    plt.show()

    print(f"Sample rate : ~{sr:.0f} Hz  ({len(gaze_df):,} total samples)")
    print(f"Confidence  : mean {g_win['confidence'].mean():.3f}")
    print(f"Velocity    : {vel_win.min():.1f} – {vel_win.max():.1f} °/s")
    print(f"X range     : {xs_sm.min():.1f} – {xs_sm.max():.1f} °")
    print(f"Y range     : {ys_sm.min():.1f} – {ys_sm.max():.1f} °")


---
## 2 Parse Trial Windows from Annotations

In [ ]:
def parse_onset_label(label):
    """
    Parse VISION_task onset annotation.
    Format: trial_001_onset_ax0_pos3R_circle
    Returns dict or None.
    """
    m = re.match(r'trial_(\d+)_onset_ax(\d+)_pos(\d+)([LR])_(\w+)', label)
    if m:
        return {
            'trial_num'   : int(m.group(1)),
            'axis_deg'    : int(m.group(2)),
            'position_num': int(m.group(3)),
            'side'        : m.group(4),
            'shape'       : m.group(5),
        }
    return None


def parse_offset_label(label):
    """Parse offset annotation → dict with trial_num and outcome."""
    m = re.match(r'trial_(\d+)_offset_(\w+)', label)
    if m:
        return {'trial_num': int(m.group(1)), 'outcome': m.group(2)}
    return None


onsets  = ann_df[ann_df['label'].str.contains('_onset_', na=False)].copy()
offsets = ann_df[ann_df['label'].str.contains('_offset_', na=False)].copy()

trial_rows = []
for _, onset_row in onsets.iterrows():
    parsed = parse_onset_label(onset_row['label'])
    if parsed is None:
        continue
    t_start = onset_row['timestamp']
    # Find matching offset (same trial_num)
    off_cands = offsets[offsets['timestamp'] > t_start]
    t_end, outcome = np.nan, 'unknown'
    for _, off_row in off_cands.iterrows():
        poff = parse_offset_label(off_row['label'])
        if poff and poff['trial_num'] == parsed['trial_num']:
            t_end   = off_row['timestamp']
            outcome = poff['outcome']
            break
    parsed['t_start']  = t_start
    parsed['t_end']    = t_end
    parsed['outcome']  = outcome
    parsed['duration_s'] = t_end - t_start if not np.isnan(t_end) else np.nan
    trial_rows.append(parsed)

trials = pd.DataFrame(trial_rows).sort_values('trial_num').reset_index(drop=True)

AXIS_LABELS = {0: 'Horizontal', 45: 'Diagonal BL→TR', 135: 'Diagonal TL→BR'}
trials['axis_label'] = trials['axis_deg'].map(AXIS_LABELS)
trials['hemifield']  = trials['side'].map({'R': 'Right', 'L': 'Left'})

print(f'Trials parsed        : {len(trials)}')
print(f'Correct (annotation) : {(trials["outcome"]=="correct").sum()}')
print(f'Incorrect            : {(trials["outcome"]=="incorrect").sum()}')
print(f'Mean trial duration  : {trials["duration_s"].mean():.3f} s')
trials.head(8)

---
## 3 Saccade Detection (Velocity Threshold)

Saccades are detected from raw gaze using a velocity-threshold method.  
A saccade is a continuous period where instantaneous velocity exceeds **30 °/s**, lasting 10–100 ms.

In [ ]:
def detect_saccades(gaze_df,
                    vel_threshold  = VELOCITY_THRESHOLD_DEG_S,
                    min_dur_s      = MIN_SACCADE_DUR_S,
                    max_dur_s      = MAX_SACCADE_DUR_S,
                    px_per_deg     = 1.0):
    """
    Detect saccades using a velocity-threshold (IVT) algorithm.
    Returns a DataFrame with columns: start, end, duration_s, amplitude_deg, peak_velocity_deg_s
    """
    ts  = gaze_df['timestamp'].values
    nx  = gaze_df['norm_pos_x'].values
    ny  = gaze_df['norm_pos_y'].values

    # ── TODO 3: Compute instantaneous angular velocity ────────────────────────
    # Steps:
    # 1. Compute dt = difference between consecutive timestamps (np.diff)
    # 2. Compute dx, dy = differences in x and y gaze position
    # 3. Compute angular displacement per sample: angle = sqrt(dx^2 + dy^2)
    # 4. Velocity (deg/s) = angle / dt
    # Pad velocity array to same length as ts (prepend 0).

    dt  = ???   # np.diff(ts)
    dx  = ???
    dy  = ???
    vel = ???   # angular velocity in degrees/second

    # ── Keep saccade detection logic below (do not modify) ───────────────────
    above = (vel >= vel_threshold).astype(int)
    edges = np.diff(above, prepend=0)
    starts = np.where(edges == 1)[0]
    ends   = np.where(edges == -1)[0]
    if len(ends) > 0 and len(starts) > 0 and ends[0] < starts[0]:
        ends = ends[1:]
    n = min(len(starts), len(ends))
    starts, ends = starts[:n], ends[:n]

    rows = []
    for s, e in zip(starts, ends):
        dur = ts[e] - ts[s]
        if not (min_dur_s <= dur <= max_dur_s):
            continue
        amp = np.sqrt((nx[e]-nx[s])**2 + (ny[e]-ny[s])**2)
        rows.append({'start': ts[s], 'end': ts[e], 'duration_s': dur,
                     'amplitude_deg': amp, 'peak_velocity_deg_s': vel[s:e+1].max()})
    return pd.DataFrame(rows) if rows else pd.DataFrame(
        columns=['start','end','duration_s','amplitude_deg','peak_velocity_deg_s'])


---
## 4 Blink Preprocessing

In [ ]:
def build_blink_table(blink_df, group_gap_s=BLINK_GROUP_GAP_S):
    onsets  = blink_df[blink_df['type'] == 'onset'].sort_values('timestamp')
    offsets = blink_df[blink_df['type'] == 'offset'].sort_values('timestamp')
    if len(onsets) == 0:
        return pd.DataFrame()
    ts_arr = onsets['timestamp'].values
    grouped_onsets = []
    group_start = ts_arr[0]
    for i in range(1, len(ts_arr)):
        if ts_arr[i] - ts_arr[i - 1] > group_gap_s:
            grouped_onsets.append(group_start)
            group_start = ts_arr[i]
    grouped_onsets.append(group_start)

    rows = []
    off_ts = offsets['timestamp'].values
    for go in grouped_onsets:
        candidates = off_ts[off_ts > go]
        if len(candidates):
            t_off = candidates[0]
            rows.append({
                'timestamp'     : go,
                'end_timestamp' : t_off,
                'duration_ms'   : (t_off - go) * 1000,
            })
    return pd.DataFrame(rows)


blink_table = build_blink_table(blink_df)
print(f'Blinks: {len(blink_table)}')
if len(blink_table):
    t_exp_start = ann_df[ann_df['label'] == 'experiment_start']['timestamp'].values
    t_exp_end   = ann_df[ann_df['label'] == 'experiment_end']['timestamp'].values
    if len(t_exp_start) and len(t_exp_end):
        exp_dur_min = (t_exp_end[0] - t_exp_start[0]) / 60
        blink_rate  = len(blink_table) / exp_dur_min
        print(f'Session duration  : {exp_dur_min:.1f} min')
        print(f'Blink rate        : {blink_rate:.1f} blinks/min')
    print(f'Mean duration     : {blink_table["duration_ms"].mean():.0f} ms')

---
## 5 Per-Trial Metric Extraction

In [ ]:
def events_in_window(df, t_start, t_end, ts_col='timestamp'):
    return df[(df[ts_col] >= t_start) & (df[ts_col] <= t_end)]

def trial_fixation_metrics(fix_in):
    if len(fix_in) == 0:
        return {
            'fix_count': 0,
            'fix_dur_mean_ms': np.nan,
            'fix_dur_total_ms': 0,
            'fix_dispersion_mean': np.nan
        }

    # FILTER: minimum duration + minimum confidence
    fix_valid = fix_in[
        (fix_in['duration_ms'] >= MIN_FIX_DURATION_MS) &
        (fix_in['confidence']  >= MIN_GAZE_CONFIDENCE)
    ]

    if len(fix_valid) == 0:
        return {
            'fix_count': 0,
            'fix_dur_mean_ms': np.nan,
            'fix_dur_total_ms': 0,
            'fix_dispersion_mean': np.nan
        }

    return {
        'fix_count': len(fix_valid),
        'fix_dur_mean_ms': fix_valid['duration_ms'].mean(),
        'fix_dur_total_ms': fix_valid['duration_ms'].sum(),
        'fix_dispersion_mean': (
            fix_valid['dispersion'].mean()
            if 'dispersion' in fix_valid else np.nan
        ),
    }


def trial_saccade_metrics(sacc_in, t_start, target_x=None, target_y=None):
    if len(sacc_in) == 0:
        return {'sacc_count': 0, 'sacc_amp_mean_deg': np.nan,
                'sacc_peak_vel_mean': np.nan, 'sacc_dur_mean_ms': np.nan,
                'sacc_latency_ms': np.nan}
    metrics = {
        'sacc_count'       : len(sacc_in),
        'sacc_amp_mean_deg': sacc_in['amplitude_deg'].mean(),
        'sacc_peak_vel_mean': sacc_in['peak_vel_deg_s'].mean(),
        'sacc_dur_mean_ms' : sacc_in['duration_ms'].mean(),
        'sacc_latency_ms'  : (sacc_in['timestamp'].min() - t_start) * 1000,
    }
    # First saccade towards target
    if target_x is not None and len(sacc_in):
        first = sacc_in.sort_values('timestamp').iloc[0]
        dist_start = np.hypot(first['start_x'] - target_x, first['start_y'] - target_y)
        dist_end   = np.hypot(first['end_x']   - target_x, first['end_y']   - target_y)
        metrics['first_sacc_toward_target'] = int(dist_end < dist_start)
    return metrics


def gaze_at_response(gaze_in, t_end, window_s=0.15):
    """
    Return median gaze position in the 150 ms window just before key press.
    Only uses high-confidence, on-screen gaze samples.
    """
    if t_end is None or np.isnan(t_end):
        return {'resp_gaze_x': np.nan, 'resp_gaze_y': np.nan, 'resp_gaze_conf': np.nan}
    window = gaze_in[
        (gaze_in['timestamp'] >= t_end - window_s) &
        (gaze_in['timestamp'] <= t_end) &
        (gaze_in['confidence'] >= MIN_GAZE_CONFIDENCE) &
        # Extra guard: keep only on-screen gaze (norm_pos in [0,1])
        (gaze_in['norm_x'].between(0.0, 1.0)) &
        (gaze_in['norm_y'].between(0.0, 1.0))
    ]
    if len(window) == 0:
        return {'resp_gaze_x': np.nan, 'resp_gaze_y': np.nan, 'resp_gaze_conf': np.nan}
    return {
        'resp_gaze_x'   : window['x_deg'].median(),
        'resp_gaze_y'   : window['y_deg'].median(),
        'resp_gaze_conf': window['confidence'].mean(),
    }


# ── Eccentricity mapping (from VISION_task.py constants) ─────────────────────
POSITION_STEP_DEG = 4.5
POSITIONS_DEG     = [POSITION_STEP_DEG * i for i in range(1, 8)]

def get_target_xy(axis_deg, position_num, side_char):
    side  = 1 if side_char == 'R' else -1
    ecc   = POSITIONS_DEG[position_num - 1]
    a_rad = np.radians(axis_deg)
    return side * ecc * np.cos(a_rad), side * ecc * np.sin(a_rad)


# ── Compute trial metrics ─────────────────────────────────────────────────────
rows = []
for _, tr in trials.iterrows():
    t0, t1 = tr['t_start'], tr['t_end']
    if np.isnan(t1):
        continue
    tx, ty  = get_target_xy(tr['axis_deg'], tr['position_num'], tr['side'])
    fix_in  = events_in_window(fix_df,    t0, t1)
    sacc_in = events_in_window(sacc_df,   t0, t1) if len(sacc_df) else pd.DataFrame()
    blink_in= events_in_window(blink_table, t0, t1) if len(blink_table) else pd.DataFrame()
    gaze_in = events_in_window(gaze_df,   t0, t1)

    row = {
        'trial_num'   : tr['trial_num'],
        'axis_deg'    : tr['axis_deg'],
        'axis_label'  : tr['axis_label'],
        'position_num': tr['position_num'],
        'eccentricity_deg': POSITIONS_DEG[tr['position_num'] - 1],
        'side'        : tr['side'],
        'hemifield'   : tr['hemifield'],
        'shape'       : tr['shape'],
        'target_x_deg': tx,
        'target_y_deg': ty,
        't_start'     : t0,
        't_end'       : t1,
        'duration_s'  : tr['duration_s'],
        'outcome'     : tr['outcome'],
        'blink_count'     : len(blink_in),
        'blink_dur_mean_ms': blink_in['duration_ms'].mean() if len(blink_in) else np.nan,
    }
    row.update(trial_fixation_metrics(fix_in))
    row.update(trial_saccade_metrics(sacc_in, t0, tx, ty))
    row.update(gaze_at_response(gaze_in, t1))

    # Gaze error at moment of response vs target position
    if not np.isnan(row.get('resp_gaze_x', np.nan)):
        row['gaze_error_deg'] = np.hypot(
            row['resp_gaze_x'] - tx, row['resp_gaze_y'] - ty
        )
    else:
        row['gaze_error_deg'] = np.nan

    rows.append(row)

trial_metrics = pd.DataFrame(rows)
print(f'Trial metrics computed: {len(trial_metrics)} rows')
trial_metrics.head()

---
## 6 Session-Level Summary

In [ ]:
tm = trial_metrics

def fmt(series, decimals=2):
    return f'{series.mean():.{decimals}f} ± {series.std():.{decimals}f}'

n_trials    = len(tm)
n_correct   = (tm['outcome'] == 'correct').sum()
accuracy    = n_correct / n_trials if n_trials else 0
exp_dur_s   = (tm['t_end'].max() - tm['t_start'].min()) if n_trials else np.nan

summary = {
    'Participant'                              : PARTICIPANT_ID,
    'Trials'                                   : n_trials,
    'Correct / Accuracy'                       : f'{n_correct} / {accuracy:.1%}',
    'Session duration (min)'                   : f'{exp_dur_s/60:.1f}' if not np.isnan(exp_dur_s) else 'n/a',
    # Gaze sample rate
    'Gaze sample rate (Hz)'                    : f'{1/np.median(np.diff(gaze_df["timestamp"].values)):.0f}',
    # Fixations
    'Fixations per trial (mean ± SD)'          : fmt(tm['fix_count'], 1),
    'Mean fixation duration (ms)'              : fmt(tm['fix_dur_mean_ms']),
    'Total fixation time per trial (ms)'       : fmt(tm['fix_dur_total_ms']),
    'Mean fixation dispersion (°)'             : fmt(tm['fix_dispersion_mean']),
    # Saccades
    'Saccades per trial (mean ± SD)'           : fmt(tm['sacc_count'], 1),
    'Mean saccade amplitude (°)'               : fmt(tm['sacc_amp_mean_deg']),
    'Mean saccade peak velocity (°/s)'         : fmt(tm['sacc_peak_vel_mean']),
    'Mean saccade latency (ms)'                : fmt(tm['sacc_latency_ms']),
    # Blinks
    'Blinks per trial (mean ± SD)'             : fmt(tm['blink_count'], 1),
    'Mean blink duration (ms)'                 : fmt(tm['blink_dur_mean_ms']),
    # Gaze accuracy at response
    'Mean gaze error at key press (°)'         : fmt(tm['gaze_error_deg']),
}

print('='*55)
for k, v in summary.items():
    print(f'  {k:<42}  {v}')
print('='*55)

---
## 9  Eye-Movement Metrics by Axis

How do saccade and blink patterns differ across the three scanning axes? Each bar shows the mean ± SEM across trials for that axis.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

metrics_to_plot = [
    ('sacc_count',         'Saccade count'),
    ('sacc_amp_mean_deg',  'Mean saccade amplitude (°)'),
    ('sacc_peak_vel_mean', 'Mean saccade peak velocity (°/s)'),
    ('sacc_latency_ms',    'First saccade latency (ms)'),
    ('blink_count',        'Blinks per trial'),
]

axis_order = ['Horizontal', 'Diagonal BL→TR', 'Diagonal TL→BR']

for ax, (metric, label) in zip(axes.flat, metrics_to_plot):
    grp = trial_metrics.groupby('axis_label')[metric]
    means = grp.mean().reindex(axis_order)
    sems  = grp.sem().reindex(axis_order)
    x = np.arange(len(axis_order))
    ax.bar(x, means, yerr=sems, capsize=6, color=['#2196F3','#FF9800','#4CAF50'], alpha=0.8)
    ax.set_xticks(x)

    # ── TODO 4: Add axis labels and title ────────────────────────────────────
    # Replace '???' with appropriate strings.
    ax.set_xticklabels(axis_order, rotation=15, ha='right')
    ax.set_xlabel('???')         # what does the x-axis represent?
    ax.set_ylabel('???')         # what does the y-axis represent?
    ax.set_title('???')          # give this subplot a descriptive title
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('???', fontsize=14, fontweight='bold')   # give the whole figure a title
plt.tight_layout()
plt.show()


---
## 10 Main Sequence (Saccade Amplitude vs Peak Velocity)

In [ ]:
if len(sacc_df) > 5:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    ax.scatter(sacc_df['amplitude_deg'], sacc_df['peak_vel_deg_s'],
               alpha=0.35, s=15, color='steelblue')
    valid = sacc_df[sacc_df['amplitude_deg'] > 0.01].copy()
    log_amp = np.log(valid['amplitude_deg'])
    log_vel = np.log(valid['peak_vel_deg_s'].clip(lower=0.01))
    mask = np.isfinite(log_amp) & np.isfinite(log_vel)
    if mask.sum() > 3:
        slope, intercept, r, p, _ = stats.linregress(log_amp[mask], log_vel[mask])
        x_fit = np.linspace(valid['amplitude_deg'].min(), valid['amplitude_deg'].max(), 100)
        ax.plot(x_fit, np.exp(intercept) * x_fit**slope, 'r-', lw=1.5,
                label=f'Power law: v = {np.exp(intercept):.1f}·amp^{slope:.2f}\nr={r:.2f}, p={p:.3f}')
    ax.set_xlabel('Saccade amplitude (°)')
    ax.set_ylabel('Peak velocity (°/s)')
    ax.set_title('Main sequence')
    ax.legend(fontsize=9)

    ax = axes[1]
    ax.scatter(sacc_df['amplitude_deg'], sacc_df['duration_ms'],
               alpha=0.35, s=15, color='darkorange')
    ax.set_xlabel('Saccade amplitude (°)')
    ax.set_ylabel('Saccade duration (ms)')
    ax.set_title('Amplitude vs Duration')

    plt.suptitle(f'{PARTICIPANT_ID} — Saccade Main Sequence', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough saccades for main sequence plot.')

---
## 11 Gaze Spatial Map at Key Press

Shows where the participant was gazing (within 150 ms before key press) vs the true target location.

In [ ]:
# ── Gaze accuracy at response: where were participants looking at key press? ──
valid = trial_metrics.dropna(subset=['resp_gaze_x', 'gaze_error_deg'])

if len(valid) == 0:
    print("No valid gaze-at-response data found.")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    palette = {'Horizontal': '#2196F3', 'Diagonal BL→TR': '#FF9800', 'Diagonal TL→BR': '#4CAF50'}
    for axis_lbl, grp in valid.groupby('axis_label'):
        sns.kdeplot(grp['gaze_error_deg'], ax=ax,
                    label=axis_lbl, color=palette.get(axis_lbl), fill=True, alpha=0.25, lw=1.5)
    ax.set_xlabel('Gaze error at key press (°)', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'{PARTICIPANT_ID} — Gaze error at key press by axis', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.axvline(valid['gaze_error_deg'].median(), color='grey', ls='--', lw=1,
               label=f"Median {valid['gaze_error_deg'].median():.1f}°")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"\nGaze error at key press (°):")
    print(valid.groupby('axis_label')['gaze_error_deg']
              .agg(['mean','median','std'])
              .round(2).to_string())


---
## 12 Export Trial-Level Gaze Metrics

In [ ]:
# ── TODO 5: Export trial-level metrics to CSV ────────────────────────────────
# Define which columns to export, set the output filename, and save.

export_cols = [
    'trial_num', 'axis_deg', 'axis_label', 'position_num', 'eccentricity_deg',
    'side', 'hemifield', 'shape', 'target_x_deg', 'target_y_deg',
    't_start', 't_end', 'duration_s', 'outcome',
    'fix_count', 'fix_dur_mean_ms', 'fix_dur_total_ms', 'fix_dispersion_mean',
    'sacc_count', 'sacc_amp_mean_deg', 'sacc_peak_vel_mean', 'sacc_dur_mean_ms',
    'sacc_latency_ms', 'first_sacc_toward_target',
    'blink_count', 'blink_dur_mean_ms',
    'resp_gaze_x', 'resp_gaze_y', 'resp_gaze_conf', 'gaze_error_deg',
]

# TODO: Set the output filename — include participant ID and session label
out_path = '???'   # e.g. f'{PARTICIPANT_ID}_VISION_task_gaze_metrics_{SESSION_LABEL}.csv'

available = [c for c in export_cols if c in trial_metrics.columns]
trial_metrics[available].to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({len(trial_metrics)} trials, {len(available)} columns)')
